# ParkWise plate detector (YOLOv5s)

Fine-tune `yolov5s.pt` on the merged Philippine plate dataset (`data/merged`).

**Before you start**
1. In Colab: Runtime → Change runtime type → GPU (T4 is enough).
2. On your PC, zip `agent-service/data/merged` (the folder that contains `train/`, `valid/`, `test/`, `data.yaml`).
3. Upload that zip when prompted. Do not commit or share plate images publicly beyond this private Colab session.

After training, download `best.pt` and save it locally as `agent-service/models/plate_detector.pt`.

In [ ]:
!nvidia-smi

In [ ]:
import os
from pathlib import Path

os.chdir("/content")
!git clone --depth 1 https://github.com/ultralytics/yolov5.git
%cd /content/yolov5
!pip install -q -r requirements.txt

## Upload and unpack `merged.zip`

Zip the `data/merged` folder on Windows, then run the next cell and choose the zip file.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
import shutil

uploaded = files.upload()
zip_name = next(iter(uploaded))
extract_root = Path("/content/data")
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(extract_root)

# Find the folder that contains train/images (zip may nest as merged/ or data/merged/)
candidates = list(extract_root.rglob("train"))
merged = None
for c in candidates:
    if (c / "images").is_dir():
        merged = c.parent
        break
if merged is None:
    raise FileNotFoundError("Could not find train/images inside the zip. Zip the data/merged folder.")

yaml_path = Path("/content/data/merged.yaml")
yaml_path.write_text(
    "\n".join([
        f"path: {merged.as_posix()}",
        "train: train/images",
        "val: valid/images",
        "test: test/images",
        "nc: 1",
        "names: ['license_plate']",
        "",
    ]),
    encoding="utf-8",
)
print("Dataset root:", merged)
print("Wrote", yaml_path)
print("train", len(list((merged / "train" / "images").glob("*"))))
print("valid", len(list((merged / "valid" / "images").glob("*"))))
print("test", len(list((merged / "test" / "images").glob("*"))))

## Train (img 640, batch 16, 100 epochs)

In [ ]:
%cd /content/yolov5
!python train.py --img 640 --batch 16 --epochs 100 --data /content/data/merged.yaml --weights yolov5s.pt --device 0 --project /content/runs --name plate_detector --exist-ok --patience 50 --single-cls

## Validation and test metrics (precision, recall, mAP@0.5)

In [ ]:
%cd /content/yolov5
print("=== VALID ===")
!python val.py --data /content/data/merged.yaml --weights /content/runs/plate_detector/weights/best.pt --img 640 --batch 16 --task val --device 0 --project /content/runs --name val_plates --exist-ok --single-cls
print("=== TEST ===")
!python val.py --data /content/data/merged.yaml --weights /content/runs/plate_detector/weights/best.pt --img 640 --batch 16 --task test --device 0 --project /content/runs --name test_plates --exist-ok --single-cls

## Download `best.pt`

Save the downloaded file on your PC as `agent-service/models/plate_detector.pt`.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

src = Path("/content/runs/plate_detector/weights/best.pt")
dst = Path("/content/plate_detector.pt")
shutil.copy2(src, dst)
files.download(str(dst))